In [25]:
from PIL import Image
def Brezenheim_otrezok(img, x0, y0, x1, y1, color=(255, 0, 0)):
    pixels = img.load()
    x0, y0 = int(round(x0)), int(round(y0))
    x1, y1 = int(round(x1)), int(round(y1))
    dx = abs(x1 - x0)
    dy = abs(y1 - y0)
    sx = 1 if x0 < x1 else -1
    sy = 1 if y0 < y1 else -1
    err = dx - dy
    w, h = img.size
    while True:
        if 0 <= x0 < w and 0 <= y0 < h:
            pixels[x0, y0] = color
        if x0 == x1 and y0 == y1:
            break
        e2 = err * 2
        if e2 > -dy:
            err -= dy
            x0 += sx
        if e2 < dx:
            err += dx
            y0 += sy
def draw_polygon(img, poly, color):
    if not poly: return
    for i in range(len(poly)):
        p1 = poly[i]
        p2 = poly[(i + 1) % len(poly)]
        Brezenheim_otrezok(img, p1[0], p1[1], p2[0], p2[1], color)
def draw_window(img, x_min, y_min, x_max, y_max, color=(0, 0, 255)):
    Brezenheim_otrezok(img, x_min, y_min, x_max, y_min, color)
    Brezenheim_otrezok(img, x_max, y_min, x_max, y_max, color)
    Brezenheim_otrezok(img, x_max, y_max, x_min, y_max, color)
    Brezenheim_otrezok(img, x_min, y_max, x_min, y_min, color)

7.      Реализовать алгоритмы отсечения плоских фигур прямоугольным окном: Сазерленда-Ходгмана, Вейлера-Азертона.

Алогоритм Сазерленда Ходгмана

In [26]:
def sutherland_hodgman(polygon, x_min, y_min, x_max, y_max):
    def inside_left(p):   return p[0] >= x_min
    def inside_right(p):  return p[0] <= x_max
    def inside_bottom(p): return p[1] >= y_min
    def inside_top(p):    return p[1] <= y_max

    def intersect_left(p1, p2): return (x_min, p1[1] + (p2[1] - p1[1]) * (x_min - p1[0]) / (p2[0] - p1[0]))
    def intersect_right(p1, p2): return (x_max, p1[1] + (p2[1] - p1[1]) * (x_max - p1[0]) / (p2[0] - p1[0]))
    def intersect_bottom(p1, p2): return (p1[0] + (p2[0] - p1[0]) * (y_min - p1[1]) / (p2[1] - p1[1]), y_min)
    def intersect_top(p1, p2): return (p1[0] + (p2[0] - p1[0]) * (y_max - p1[1]) / (p2[1] - p1[1]), y_max)

    clippers = [
        (inside_left, intersect_left), (inside_right, intersect_right),
        (inside_bottom, intersect_bottom), (inside_top, intersect_top)
    ]

    clipped_polygon = polygon
    for is_inside, get_intersection in clippers:
        if not clipped_polygon: break
        new_polygon = []
        n = len(clipped_polygon)
        for i in range(n):
            p1 = clipped_polygon[i]
            p2 = clipped_polygon[(i + 1) % n]
            if is_inside(p1) and is_inside(p2):
                new_polygon.append(p2)
            elif is_inside(p1) and not is_inside(p2):
                new_polygon.append(get_intersection(p1, p2))
            elif not is_inside(p1) and is_inside(p2):
                new_polygon.append(get_intersection(p1, p2))
                new_polygon.append(p2)
        clipped_polygon = new_polygon
    return clipped_polygon

In [27]:
canvas_size = (200, 200)
win = (100, 0, 150, 200)
poly = [(20, 20), (180, 20), (180, 80), (80, 80), (80, 120), (180, 120), (180, 180), (20, 180)]
img1 = Image.new("RGB", canvas_size, (255, 255, 255))
draw_polygon(img1, poly, (200, 200, 200))
draw_window(img1, *win, (0, 0, 255))
sh_poly = sutherland_hodgman(poly, *win)
if sh_poly:
    draw_polygon(img1, sh_poly, (255, 0, 0))
img1.show(title="Sutherland-Hodgman")

Алгоритм Вайлера - Азертона

In [28]:
class Node:
    def __init__(self, x, y, is_inter=False):
        self.x, self.y = x, y
        self.is_inter = is_inter
        self.is_entry = False
        self.visited = False
        self.next = None
        self.twin = None
        self.alpha = 0.0

def weiler_atherton(subj_points, x_min, y_min, x_max, y_max):
    clip_points = [(x_min, y_min), (x_max, y_min), (x_max, y_max), (x_min, y_max)]
    def inside_clip(x, y):
        return x_min <= x <= x_max and y_min <= y <= y_max
    def line_intersect(p1, p2, p3, p4):
        den = (p1[0] - p2[0]) * (p3[1] - p4[1]) - (p1[1] - p2[1]) * (p3[0] - p4[0])
        if den == 0: return None
        t = ((p1[0] - p3[0]) * (p3[1] - p4[1]) - (p1[1] - p3[1]) * (p3[0] - p4[0])) / den
        u = -((p1[0] - p2[0]) * (p1[1] - p3[1]) - (p1[1] - p2[1]) * (p1[0] - p3[0])) / den
        if 0 <= t < 1 and 0 <= u < 1:
            return (p1[0] + t * (p2[0] - p1[0]), p1[1] + t * (p2[1] - p1[1]), t, u)
        return None
    subj_list = [Node(p[0], p[1]) for p in subj_points]
    clip_list = [Node(p[0], p[1]) for p in clip_points]
    for i in range(len(subj_list)): subj_list[i].next = subj_list[(i+1)%len(subj_list)]
    for i in range(len(clip_list)): clip_list[i].next = clip_list[(i+1)%len(clip_list)]
    intersections_found = False
    for i in range(len(subj_points)):
        s_node, s_next = subj_list[i], subj_list[i].next
        p1, p2 = (s_node.x, s_node.y), (s_next.x, s_next.y)
        s_inters = []
        for j in range(4):
            c_node = clip_list[j]
            p3, p4 = clip_points[j], clip_points[(j+1)%4]
            inter = line_intersect(p1, p2, p3, p4)
            if inter:
                ix, iy, alpha_s, alpha_c = inter
                intersections_found = True
                s_in, c_in = Node(ix, iy, True), Node(ix, iy, True)
                s_in.alpha, c_in.alpha = alpha_s, alpha_c
                s_in.twin, c_in.twin = c_in, s_in
                s_inters.append((s_in, c_in, c_node))
        s_inters.sort(key=lambda item: item[0].alpha)
        start_inside = inside_clip(*p1)
        curr_s = s_node
        for s_in, c_in, c_node in s_inters:
            if not start_inside:
                s_in.is_entry = True
                start_inside = True
            else:
                s_in.is_entry = False
                start_inside = False
            s_in.next = curr_s.next
            curr_s.next = s_in
            curr_s = s_in
            curr_c = c_node
            while curr_c.next.is_inter and curr_c.next.alpha < c_in.alpha:
                curr_c = curr_c.next
            c_in.next = curr_c.next
            curr_c.next = c_in
    if not intersections_found:
        return [subj_points] if inside_clip(subj_points[0][0], subj_points[0][1]) else []
    result_polygons = []
    entry_nodes = []
    curr = subj_list[0]
    while True:
        if curr.is_inter and curr.is_entry:
            entry_nodes.append(curr)
        curr = curr.next
        if curr == subj_list[0]: break
    for start_node in entry_nodes:
        if start_node.visited: continue
        new_poly = []
        curr = start_node
        while not curr.visited:
            curr.visited = True
            new_poly.append((curr.x, curr.y))
            if curr.is_inter:
                curr.twin.visited = True
            if curr.is_inter and not curr.is_entry:
                curr = curr.twin.next
            else:
                curr = curr.next
        result_polygons.append(new_poly)
    return result_polygons

In [30]:
img2 = Image.new("RGB", canvas_size, (255, 255, 255))
draw_polygon(img2, poly, (200, 200, 200))
draw_window(img2, *win, (0, 0, 255))
wa_polys = weiler_atherton(poly, *win)
colors = [(255, 0, 0), (0, 200, 0)]
for i, p in enumerate(wa_polys):
    draw_polygon(img2, p, colors[i % len(colors)])
img2.show(title="Weiler-Atherton")